# Séquence 5 — Indicateurs, agrégations et pièges décisionnels
Parcours sans corrigé. À chaque étape: action, confiance, preuve, incertitude, limite.

In [ ]:
from pathlib import Path
import sys
ROOT=Path.cwd().resolve()
while not (ROOT/'src').exists() and ROOT!=ROOT.parent: ROOT=ROOT.parent
sys.path.insert(0,str(ROOT/'src'))
from iot_decision.indicators import load_measurements,global_mean,zone_maxima,find_masked_zones,duration_above_threshold
from iot_decision.quality import load_raw,flatten
measurements=ROOT/'data/processed/batch001_measurements.csv'

## Une moyenne peut être vraie et trompeuse
Avant de calculer : la moyenne globale de ce lot vous semble-t-elle devoir dépasser 30 °C ? Le seuil pédagogique de 35 °C vous semble-t-il concerné ?

In [ ]:
rows=load_measurements(measurements)
global_mean(rows),zone_maxima(rows)

## Quelle zone la moyenne masque-t-elle ?
Une zone peut franchir le seuil sans que la moyenne globale ne bouge. Identifiez-la, puis vérifiez depuis quand elle est réellement observée au-dessus du seuil.

In [ ]:
masked=find_masked_zones(rows,threshold=35.0)
[(m.zone,m.zone_max,m.global_mean_value) for m in masked]

In [ ]:
duration_above_threshold(rows,'battery-shelter-01',threshold=35.0)

## Un score que vous ne pouvez pas encore lire
`risk_score.py` fournit une recommandation automatique. N'ouvrez pas ce fichier avant le débrief : interrogez-le seulement comme le ferait un décideur pressé, à partir de son résultat.

In [ ]:
from iot_decision.risk_score import score_all_zones
known=[flatten(e) for e in load_raw(ROOT/'data/raw/batch001_raw.jsonl')]
for r in known: r['value']=float(r['value'])
[(e.zone,round(e.score),e.decision) for e in score_all_zones(known)]

## Le cas qui devrait vous alerter
Une nouvelle zone, `fuel-storage-01`, vient d'être équipée. Interrogez le même score sur cette zone avant de lire quoi que ce soit sur elle. Comparez son score à ceux des zones déjà connues : que vous dit cette comparaison, et que vous cache-t-elle ?

In [ ]:
shift=[flatten(e) for e in load_raw(ROOT/'data/samples/batch003_shift_scenario.jsonl')]
for r in shift: r['value']=float(r['value'])
[(e.zone,round(e.score),e.decision) for e in score_all_zones(shift)]

## Votre recommandation
Vous savez maintenant qu'une zone de stockage carburant a un seuil de sécurité réel bien plus bas que 35 °C, information que le score n'a jamais reçue. Rédigez votre recommandation : quel processus de décision présentez-vous au commandant — la moyenne, le maximum, le score automatique, ou une combinaison — et avec quelle confiance ? L'appel suivant teste seulement l'API.

In [ ]:
from iot_decision.risk_score import decide
assert decide(0) == 'aucune action requise'
assert decide(100) == 'inspection recommandée'